# Project: Network Intrusion Detection System (NIDS)
**Author:** Sukhman Singh
**Organization:** C-DAC Mohali (Summer Training)

This Google Colab notebook contains the training pipeline for the Hybrid CNN-LSTM network traffic intrusion detection model with a custom attention mechanism. 

To achieve robust generalization, this notebook uses the **NSL-KDD** benchmark dataset, processed using a standard stratified validation split strategy.

You can run this notebook directly in Google Colab to train the model using GPU acceleration. Once training completes, run the final cell to automatically download the generated files to your local computer.

### Key Engineering Details:
1. **Stratified Validation Split:** Splits the training dataset into train and validation subsets *before* scaling and SMOTE oversampling. This prevents data leakage and ensures validation metrics reflect true generalization performance.
2. **Standard Regularization:** Implements spatial dropout on Convolutional layers, standard dropout on recurrent and feedforward layers, and L2 kernel regularization to prevent overfitting.
3. **Automatic Downloader Integration:** Integrates Colab's native downloading utilities to automatically zip/download model weights (`cnn_lstm_model.keras`), scaler, encoders, metadata, and performance graphs directly to your local computer once training completes.


In [ ]:
# Install dependencies needed in Colab environment
!pip install -q imbalanced-learn scikit-learn matplotlib seaborn joblib requests

import os
# Configure Keras backend to use TensorFlow
os.environ["KERAS_BACKEND"] = "tensorflow"
print("Keras backend set to TensorFlow.")

## 2. Configuration & Parameter Setup

We define paths and hyperparameters here. The directories `data`, `models`, and `outputs` will be created automatically in Colab's workspace root.

In [ ]:
import os

# --- Directory layout inside Google Colab ---
DATA_DIR = "./data"
RAW_DATA_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed")
MODELS_DIR = "./models"
OUTPUTS_DIR = "./outputs"

for dir_path in [RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR, OUTPUTS_DIR]:
    os.makedirs(dir_path, exist_ok=True)

# --- NSL-KDD Dataset URLs and file names ---
DATASET_URLS = {
    "train": "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt",
    "test": "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt",
}

TRAIN_FILE = os.path.join(RAW_DATA_DIR, "KDDTrain+.txt")
TEST_FILE = os.path.join(RAW_DATA_DIR, "KDDTest+.txt")

# --- Column names ---
COLUMN_NAMES = [
    "duration", "protocol_type", "service", "flag", "src_bytes",
    "dst_bytes", "land", "wrong_fragment", "urgent", "hot",
    "num_failed_logins", "logged_in", "num_compromised", "root_shell",
    "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "label", "difficulty_level",
]

CATEGORICAL_FEATURES = ["protocol_type", "service", "flag"]

# --- Attack mapping ---
ATTACK_MAPPING = {
    "normal": "Normal",
    "back": "DoS", "land": "DoS", "neptune": "DoS", "pod": "DoS",
    "smurf": "DoS", "teardrop": "DoS", "apache2": "DoS", "udpstorm": "DoS",
    "processtable": "DoS", "mailbomb": "DoS",
    "ipsweep": "Probe", "nmap": "Probe", "portsweep": "Probe",
    "satan": "Probe", "mscan": "Probe", "saint": "Probe",
    "ftp_write": "R2L", "guess_passwd": "R2L", "imap": "R2L",
    "multihop": "R2L", "phf": "R2L", "spy": "R2L", "warezclient": "R2L",
    "warezmaster": "R2L", "snmpgetattack": "R2L", "named": "R2L",
    "xlock": "R2L", "xsnoop": "R2L", "sendmail": "R2L",
    "httptunnel": "R2L", "worm": "R2L", "snmpguess": "R2L",
    "buffer_overflow": "U2R", "loadmodule": "U2R", "perl": "U2R",
    "rootkit": "U2R", "xterm": "U2R", "ps": "U2R",
    "sqlattack": "U2R",
}

CLASS_NAMES = ["Normal", "DoS", "Probe", "R2L", "U2R"]
NUM_CLASSES = len(CLASS_NAMES)

# --- Hyperparameters ---
SEQUENCE_LENGTH = 10
CNN_FILTERS_1 = 64
CNN_KERNEL_SIZE_1 = 3
CNN_FILTERS_2 = 128
CNN_KERNEL_SIZE_2 = 3
POOL_SIZE = 2
LSTM_UNITS_1 = 128
LSTM_UNITS_2 = 64
DENSE_UNITS = 128
DROPOUT_RATE = 0.2  # Reduced from 0.4 to allow the model to learn better representations
L2_REGULARIZATION = 1e-5  # Reduced L2 penalty to prevent underfitting

# --- Training Configuration ---
BATCH_SIZE = 256
EPOCHS = 30  # Reduced max limit
LEARNING_RATE = 0.0005
EARLY_STOPPING_PATIENCE = 4  # Reduced to be more aggressive
REDUCE_LR_PATIENCE = 3
REDUCE_LR_FACTOR = 0.5
VALIDATION_SPLIT = 0.15

# --- Output paths inside Colab ---
MODEL_SAVE_PATH = os.path.join(MODELS_DIR, "cnn_lstm_model.keras")
SCALER_SAVE_PATH = os.path.join(MODELS_DIR, "scaler.joblib")
LABEL_ENCODERS_SAVE_PATH = os.path.join(MODELS_DIR, "label_encoders.joblib")
CLASSIFICATION_REPORT_PATH = os.path.join(OUTPUTS_DIR, "classification_report.json")
CONFUSION_MATRIX_PATH = os.path.join(OUTPUTS_DIR, "confusion_matrix.png")
TRAINING_CURVES_PATH = os.path.join(OUTPUTS_DIR, "training_curves.png")
ROC_CURVES_PATH = os.path.join(OUTPUTS_DIR, "roc_curves.png")
ATTACK_DISTRIBUTION_PATH = os.path.join(OUTPUTS_DIR, "attack_distribution.png")
MODEL_METADATA_PATH = os.path.join(MODELS_DIR, "model_metadata.json")

print("NSL-KDD configurations successfully initialized.")

## 3. Data Ingestion (Downloading & Parsing NSL-KDD dataset)

The following code downloads the NSL-KDD raw data directly from the official defcom17 UNB repository and parses it into Pandas DataFrames. It also groups the 40+ granular attack labels into 5 main categories: `Normal`, `DoS`, `Probe`, `R2L`, and `U2R`.

In [ ]:
import pandas as pd
import requests

def download_file(url: str, save_path: str) -> None:
    if os.path.exists(save_path):
        print(f"File already exists: {save_path}")
        return

    print(f"Downloading {url} ...")
    response = requests.get(url, timeout=120)
    response.raise_for_status()

    with open(save_path, "wb") as f:
        f.write(response.content)
    print(f"Saved to {save_path} ({len(response.content) / 1024:.1f} KB)")

def download_dataset() -> None:
    download_file(DATASET_URLS["train"], TRAIN_FILE)
    download_file(DATASET_URLS["test"], TEST_FILE)

def map_attack_label(label: str) -> str:
    label_clean = label.strip().lower()
    category = ATTACK_MAPPING.get(label_clean)
    if category is None:
        return "Normal"
    return category

def load_dataset(file_path: str) -> pd.DataFrame:
    print(f"Loading dataset from {file_path}")
    df = pd.read_csv(file_path, header=None, names=COLUMN_NAMES)
    df["label"] = df["label"].astype(str).str.strip().str.rstrip(".")
    df["attack_category"] = df["label"].apply(map_attack_label)
    df = df.drop(columns=["difficulty_level"], errors="ignore")
    print(f"Loaded {len(df)} records with {df['attack_category'].nunique()} attack categories")
    print(df['attack_category'].value_counts())
    return df

def load_train_test() -> tuple[pd.DataFrame, pd.DataFrame]:
    download_dataset()
    train_df = load_dataset(TRAIN_FILE)
    test_df = load_dataset(TEST_FILE)
    return train_df, test_df

## 4. Data Preprocessing (Encoding, Scaling, and SMOTE)

This block handles LabelEncoding for categorical columns, standardizing numeric columns using standard scaling, and applying a tuned SMOTE strategy to oversample the highly-imbalanced minority attack classes (U2R and R2L) to 30% of the majority class count. This avoids creating excess synthetic data that fails to generalize. We shuffle the training data right after oversampling. We also save the scalers and encoders.

In [ ]:
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import RandomOverSampler

class DataPreprocessor:
    def __init__(self):
        self.label_encoders = {}
        self.scaler = StandardScaler()
        self.target_encoder = LabelEncoder()
        self._is_fitted = False

    def fit_transform(
        self, train_df: pd.DataFrame, test_df: pd.DataFrame, val_size: float = 0.15
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        print("Starting preprocessing pipeline with balanced oversampling...")
        train = train_df.copy()
        test = test_df.copy()

        # Step 1: Drop constant features that introduce noise (e.g., num_outbound_cmds is always 0)
        constant_cols = ["num_outbound_cmds"]
        train = train.drop(columns=[c for c in constant_cols if c in train.columns], errors="ignore")
        test = test.drop(columns=[c for c in constant_cols if c in test.columns], errors="ignore")

        # Step 2: Log transform heavily skewed numeric columns (duration, src_bytes, dst_bytes)
        skewed_cols = ["duration", "src_bytes", "dst_bytes"]
        for col in skewed_cols:
            if col in train.columns:
                train[col] = np.log1p(train[col].values.astype(np.float32))
                test[col] = np.log1p(test[col].values.astype(np.float32))

        # Step 3: Encode categorical features (protocol_type, service, flag)
        for col in CATEGORICAL_FEATURES:
            le = LabelEncoder()
            all_values = pd.concat([train[col], test[col]]).unique()
            le.fit(all_values)
            train[col] = le.transform(train[col])
            test[col] = le.transform(test[col])
            self.label_encoders[col] = le

        # Step 4: Encode target labels (5 categories)
        self.target_encoder.fit(CLASS_NAMES)
        train_labels = train["attack_category"].values
        test_labels = test["attack_category"].values
        y_train = self.target_encoder.transform(train_labels)
        y_test = self.target_encoder.transform(test_labels)

        print(f"Target classes: {list(self.target_encoder.classes_)}")

        # Step 5: Extract feature columns
        drop_cols = ["label", "attack_category"]
        feature_cols = [c for c in train.columns if c not in drop_cols]

        X_train = train[feature_cols].values.astype(np.float32)
        X_test = test[feature_cols].values.astype(np.float32)

        # Split train & validation BEFORE scaling and oversampling to prevent leakage
        X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
            X_train, y_train, test_size=val_size, random_state=42, stratify=y_train
        )

        # Scale features (fit StandardScaler only on the training split)
        self.scaler.fit(X_train_split)
        X_train_split = self.scaler.transform(X_train_split)
        X_val_split = self.scaler.transform(X_val_split)
        X_test = self.scaler.transform(X_test)

        print(f"Scaled features: {X_train_split.shape[1]} columns")

        # Step 6: Apply RandomOverSampler to balance minority classes
        # Calculate dynamic target sampling strategy to oversample minority classes
        # to 30% of the majority class count
        unique_classes, class_counts = np.unique(y_train_split, return_counts=True)
        majority_class_count = max(class_counts)
        target_count = int(majority_class_count * 0.3)
        
        sampling_strategy = {}
        for cls, count in zip(unique_classes, class_counts):
            if count < target_count:
                sampling_strategy[int(cls)] = target_count
            else:
                sampling_strategy[int(cls)] = count
                
        print(f"Applying RandomOverSampler with strategy: {sampling_strategy}")
        ros = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=42)
        X_train_resampled, y_train_resampled = ros.fit_resample(X_train_split, y_train_split)

        # Step 7: Shuffle training split
        shuffle_idx = np.random.RandomState(42).permutation(len(X_train_resampled))
        X_train_resampled = X_train_resampled[shuffle_idx]
        y_train_resampled = y_train_resampled[shuffle_idx]

        self._is_fitted = True
        print(f"Preprocessing complete. Train resampled: {X_train_resampled.shape}, Val: {X_val_split.shape}, Test: {X_test.shape}")
        print(f"Training Class Distribution: {dict(zip(*np.unique(y_train_resampled, return_counts=True)))}")

        return X_train_resampled, y_train_resampled, X_val_split, y_val_split, X_test, y_test

## 5. Sliding Window Sequence Construction

To leverage both spatial (feature relations) and temporal learning (attack progression over time), we group flat records into overlapping sequences of 10 consecutive connection records using a sliding window. The classification target for each sequence is the label of the final record in the window. We also one-hot encode the output targets.

In [ ]:
from tensorflow.keras.utils import to_categorical

def build_sequences(X: np.ndarray, y: np.ndarray, sequence_length: int = SEQUENCE_LENGTH) -> tuple[np.ndarray, np.ndarray]:
    num_samples, num_features = X.shape
    if num_samples < sequence_length:
        raise ValueError(f"Not enough samples ({num_samples}) to build sequences of length {sequence_length}")
    num_sequences = num_samples - sequence_length + 1

    print(f"Building sequences: {num_samples} records -> {num_sequences} sequences of length {sequence_length}")

    X_seq = np.empty((num_sequences, sequence_length, num_features), dtype=np.float32)
    y_seq_raw = np.empty(num_sequences, dtype=y.dtype)

    for i in range(num_sequences):
        X_seq[i] = X[i : i + sequence_length]
        y_seq_raw[i] = y[i + sequence_length - 1]

    y_seq = to_categorical(y_seq_raw, num_classes=NUM_CLASSES)
    print(f"Sequence shapes -> X: {X_seq.shape}, y: {y_seq.shape}")
    return X_seq, y_seq

## 6. Model Definition (Hybrid CNN-LSTM-Attention)

We define a Custom Attention Layer that learns feature importances at each step, and construct the model architecture. 1D CNNs extract spatial relations inside single records, Bidirectional LSTMs model temporal correlations across sequential network connections, and Attention targets important events.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

class AttentionLayer(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.W_attention = self.add_weight(
            name="attention_weight",
            shape=(feature_dim, feature_dim),
            initializer="glorot_uniform",
            trainable=True,
        )
        self.b_attention = self.add_weight(
            name="attention_bias",
            shape=(feature_dim,),
            initializer="zeros",
            trainable=True,
        )
        self.v_attention = self.add_weight(
            name="attention_vector",
            shape=(feature_dim, 1),
            initializer="glorot_uniform",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs):
        score = tf.nn.tanh(
            tf.tensordot(inputs, self.W_attention, axes=[[2], [0]]) + self.b_attention
        )
        attention_weights = tf.nn.softmax(
            tf.tensordot(score, self.v_attention, axes=[[2], [0]]), axis=1
        )
        context = tf.reduce_sum(inputs * attention_weights, axis=1)
        return context, attention_weights

    def get_config(self):
        return super().get_config()

def build_model(input_shape: tuple, num_classes: int = NUM_CLASSES) -> Model:
    print(f"Building model with input_shape={input_shape}, classes={num_classes}")
    inputs = layers.Input(shape=input_shape, name="input_sequence")

    l2_reg = tf.keras.regularizers.l2(L2_REGULARIZATION)

    # CNN Block: Extracts spatial correlations inside single packets
    x = layers.Conv1D(filters=CNN_FILTERS_1, kernel_size=CNN_KERNEL_SIZE_1, padding="same", kernel_regularizer=l2_reg, name="conv1d_1")(inputs)
    x = layers.BatchNormalization(name="bn_1")(x)
    x = layers.Activation("relu", name="relu_1")(x)
    x = layers.SpatialDropout1D(0.2, name="spatial_dropout_1")(x)  # Prevents early spatial overfitting

    x = layers.Conv1D(filters=CNN_FILTERS_2, kernel_size=CNN_KERNEL_SIZE_2, padding="same", kernel_regularizer=l2_reg, name="conv1d_2")(x)
    x = layers.BatchNormalization(name="bn_2")(x)
    x = layers.Activation("relu", name="relu_2")(x)
    x = layers.MaxPooling1D(pool_size=POOL_SIZE, name="maxpool")(x)
    x = layers.SpatialDropout1D(0.2, name="spatial_dropout_2")(x)

    # LSTM Block: Learns temporal attack sequence correlations
    x = layers.Bidirectional(layers.LSTM(LSTM_UNITS_1, return_sequences=True, name="lstm_1"), name="bilstm")(x)
    x = layers.Dropout(DROPOUT_RATE, name="dropout_lstm")(x)

    # Attention Block: Learns to weight key historical time steps
    context, attention_weights = AttentionLayer(name="attention")(x)

    # Classification Head with L2 Regularization & Dropout
    x = layers.Dense(DENSE_UNITS, activation="relu", kernel_regularizer=l2_reg, name="dense_1")(context)
    x = layers.Dropout(DROPOUT_RATE, name="dropout_dense")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="output")(x)

    model = Model(inputs=inputs, outputs=outputs, name="CNN_LSTM_Attention_IDS")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    model.summary()
    return model

## 7. Model Training with Class Weight Boosting & Early Stopping

The training phase uses custom class weight adjustments to address class imbalance for minor classes. It implements early stopping, learning rate scaling on plateaus, and periodic model checkpointing, saving the best `.keras` model weights directly to the Colab files storage.

In [ ]:
import json
from datetime import datetime
from sklearn.utils.class_weight import compute_class_weight

def compute_weights(y_one_hot: np.ndarray) -> dict:
    y_int = np.argmax(y_one_hot, axis=1)
    unique_classes = np.unique(y_int)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=unique_classes,
        y=y_int,
    )
    class_weights = {int(cls): float(w) for cls, w in zip(unique_classes, weights)}

    print(f"Final Class Weights: {class_weights}")
    return class_weights


class StopAtAccuracyThreshold(tf.keras.callbacks.Callback):
    """Custom callback to halt training when a specific training accuracy is reached, avoiding overfitting."""
    def __init__(self, threshold=0.96):
        super().__init__()
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            logs = {}
        accuracy = logs.get('accuracy')
        if accuracy is not None and accuracy >= self.threshold:
            print(f"\nEpoch {epoch+1}: Reached {accuracy*100:.2f}% training accuracy, which meets the threshold of {self.threshold*100:.2f}%. Stopping training to prevent overfitting!")
            self.model.stop_training = True


def get_callbacks() -> list:
    return [
        StopAtAccuracyThreshold(threshold=0.96),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=MODEL_SAVE_PATH,
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=REDUCE_LR_FACTOR,
            patience=REDUCE_LR_PATIENCE,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

def train_model(model, X_train, y_train, X_val, y_val):
    print("=" * 60)
    print("STARTING MODEL TRAINING")
    print("=" * 60)
    print(f"Training samples: {X_train.shape[0]}")
    print(f"Validation samples: {X_val.shape[0]}")

    class_weights = compute_weights(y_train)

    callbacks = get_callbacks()
    start_time = datetime.now()

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1,
    )

    training_time = (datetime.now() - start_time).total_seconds()
    print(f"Training completed in {training_time:.1f} seconds")

    # Save history
    history_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    history_data["training_time_seconds"] = training_time
    history_data["total_epochs"] = len(history.history["loss"])

    with open(MODEL_METADATA_PATH, "w") as f:
        json.dump({
            "model_name": "CNN_LSTM_Attention_IDS",
            "trained_at": datetime.now().isoformat(),
            "training_time_seconds": training_time,
            "total_epochs_trained": len(history.history["loss"]),
            "input_shape": list(X_train.shape[1:]),
            "num_classes": NUM_CLASSES,
            "class_names": CLASS_NAMES,
            "sequence_length": SEQUENCE_LENGTH,
            "dataset_used": "nsl-kdd"
        }, f, indent=2)

    return history_data

## 8. Evaluation & Visualization Generation

This section generates classification reports, Confusion Matrices (raw & normalized), ROC curves, and training history curves. It saves all visualizations as PNG files inside the `./outputs` folder and outputs them directly to the notebook screen for immediate feedback.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score, f1_score

def evaluate_model(model, X_test, y_test):
    print("=" * 60)
    print("EVALUATING MODEL")
    print("=" * 60)

    y_pred_proba = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = np.argmax(y_test, axis=1)

    # Classification report
    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    overall_acc = accuracy_score(y_true, y_pred)
    overall_f1 = f1_score(y_true, y_pred, average="weighted")
    print(f"Overall Accuracy: {overall_acc:.4f}")
    print(f"Weighted F1 Score: {overall_f1:.4f}")

    report["overall_accuracy"] = overall_acc
    report["weighted_f1"] = overall_f1

    with open(CLASSIFICATION_REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)

    # Plot Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    cm_normalized = cm.astype("float") / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
    axes[0].set_title("Confusion Matrix (Counts)")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")

    sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
    axes[1].set_title("Confusion Matrix (Normalized)")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Actual")

    plt.tight_layout()
    plt.savefig(CONFUSION_MATRIX_PATH, dpi=150, bbox_inches="tight")
    plt.show()

    # Plot ROC curves
    fig, ax = plt.subplots(figsize=(10, 8))
    for i, class_name in enumerate(CLASS_NAMES):
        if y_test[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_test[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=2, label=f"{class_name} (AUC = {roc_auc:.3f})")

    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random Classifier")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curves (One-vs-Rest)")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)
    plt.savefig(ROC_CURVES_PATH, dpi=150, bbox_inches="tight")
    plt.show()

def plot_history_curves(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history["loss"]) + 1)

    axes[0].plot(epochs, history["loss"], "b-", linewidth=2, label="Training Loss")
    if "val_loss" in history:
        axes[0].plot(epochs, history["val_loss"], "r-", linewidth=2, label="Validation Loss")
    axes[0].set_title("Model Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history["accuracy"], "b-", linewidth=2, label="Training Accuracy")
    if "val_accuracy" in history:
        axes[1].plot(epochs, history["val_accuracy"], "r-", linewidth=2, label="Validation Accuracy")
    axes[1].set_title("Model Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(TRAINING_CURVES_PATH, dpi=150, bbox_inches="tight")
    plt.show()

## 9. Running the Training Pipeline

This block runs the entire training pipeline. It loads the dataset, preprocesses it, builds the temporal sequences, instantiates the hybrid CNN-LSTM model, trains it with early stopping, and generates evaluation reports and plots.

In [ ]:
# 1. Download & load NSL-KDD dataset
train_df, test_df = load_train_test()

# 2. Preprocess data (Split train & val BEFORE SMOTE to prevent data leakage)
preprocessor = DataPreprocessor()
X_train, y_train, X_val, y_val, X_test, y_test = preprocessor.fit_transform(train_df, test_df, val_size=VALIDATION_SPLIT)
preprocessor.save_transformers()

# 3. Build sequences for train, validation, and test splits
X_train_seq, y_train_seq = build_sequences(X_train, y_train)
X_val_seq, y_val_seq = build_sequences(X_val, y_val)

shuffle_idx = np.random.RandomState(99).permutation(len(X_test))
X_test_shuffled = X_test[shuffle_idx]
y_test_shuffled = y_test[shuffle_idx]
X_test_seq, y_test_seq = build_sequences(X_test_shuffled, y_test_shuffled)

# 4. Build and train model
input_shape = (X_train_seq.shape[1], X_train_seq.shape[2])
model = build_model(input_shape)

# Train using separate validation sequences (prevents validation metrics inflation)
history = train_model(model, X_train_seq, y_train_seq, X_val_seq, y_val_seq)

# 5. Evaluate on independent test set and plot
evaluate_model(model, X_test_seq, y_test_seq)
plot_history_curves(history)

## 10. Next Steps: Import Trained Model to Local Project

Once the execution completes successfully:
1. Open the File Explorer tab on the left side of Google Colab.
2. Download the following files:
   - `models/cnn_lstm_model.keras`
   - `models/scaler.joblib`
   - `models/label_encoders.joblib`
   - `models/model_metadata.json`
3. Download any files in the `outputs/` directory if you want to update the local dashboard plots and metrics (e.g., `classification_report.json`, `confusion_matrix.png`, etc.).
4. Place these downloaded files into the respective `models/` and `outputs/` directories of your local `cdac-project` folder on your computer.
5. Once imported, you can run the local Flask API or CLI predictions directly:
   - `python run_pipeline.py --mode predict`
   - `python run_pipeline.py --mode api`

In [ ]:
# Run this cell after training finishes to automatically download all
# trained weights, scaler, encoders, and performance curves to your computer.
try:
    from google.colab import files
    import time

    print("Preparing downloads...")
    files_to_download = [
        SCALER_SAVE_PATH,
        LABEL_ENCODERS_SAVE_PATH,
        MODEL_SAVE_PATH,
        MODEL_METADATA_PATH,
        CLASSIFICATION_REPORT_PATH,
        CONFUSION_MATRIX_PATH,
        TRAINING_CURVES_PATH,
        ROC_CURVES_PATH,
        ATTACK_DISTRIBUTION_PATH
    ]

    for f_path in files_to_download:
        if os.path.exists(f_path):
            print(f"Downloading: {f_path}")
            files.download(f_path)
            time.sleep(1) # Sleep to avoid blocking download queues
except ImportError:
    print("Not running in Google Colab. Skipping automatic download.")